# Exploring one table: distributions, odd rows, and relationships

**Master's in Business Data Science · Module 1 · Session 03, part 1**

Last week you built a table and protected a join. You know what one row is, you know the
counts, and you have never once looked at the shape of a single column.

That is where today starts. Before any comparison, any test, any chart that goes in front
of somebody: what does this data actually look like, and which rows should not be in it.

Nothing in this notebook is a new dataset. It is the same Spotify snapshot and the same
table you built in session 02, which is the point. Exploratory work is not a phase you do
once at the start. It is what you do every time you are about to make a claim.

## What we do today, part 1

| | |
|---|---|
| 1 | The table you built last week, and what one row still means |
| 2 | Distributions, and what a mean hides |
| 3 | Rows that cannot be right |
| 4 | Categories and base rates |
| 5 | Relationships between columns |


In [ ]:
# Setup. Same data file as session 02, same loading pattern.
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

print(f"pandas {pd.__version__}")

SONGS_URL = "https://raw.githubusercontent.com/aaubs/ds-master/codex/m1-pandas-2026/data/M1_2026/spotify_songs.csv"
candidates = [Path("data/M1_2026/spotify_songs.csv"), Path("../data/M1_2026/spotify_songs.csv"),
              Path("ds-master/data/M1_2026/spotify_songs.csv"), Path("../ds-master/data/M1_2026/spotify_songs.csv")]

songs = None
for path in candidates:
    if path.exists():
        songs = pd.read_csv(path)
        print(f"Loaded the class snapshot from {path}")
        break
if songs is None:
    songs = pd.read_csv(SONGS_URL)
    print("Loaded the class snapshot from the class URL")

RENAMES = {"track_name": "title", "track_artist": "artist", "track_popularity": "popularity",
           "playlist_genre": "genre", "playlist_subgenre": "subgenre"}
TRACK_COLUMNS = ["track_id", "title", "artist", "energy", "danceability", "loudness", "valence",
                 "tempo", "duration_ms", "acousticness", "instrumentalness", "speechiness",
                 "popularity", "mode"]

print(f"Raw shape: {songs.shape[0]:,} rows x {songs.shape[1]} columns")


### The table from session 02

One cell, no new ideas. If any of it is unfamiliar, the session 02 notebook is where it is explained.


In [ ]:
# Last week's table, rebuilt in one cell. Nothing here is new.
# One row = one recorded track-playlist-genre association, exactly as in session 02.
membership = songs.rename(columns=RENAMES)[["track_id", "playlist_id", "genre"]].drop_duplicates()
tracks = songs.rename(columns=RENAMES).drop_duplicates(subset=["track_id"])[TRACK_COLUMNS]

joined = membership.merge(tracks, on="track_id", how="left", validate="many_to_one")
assert len(joined) == len(membership)

print(f"Associations (one row each): {len(joined):,}")
print(f"Distinct tracks behind them: {tracks['track_id'].nunique():,}")


> **Judgement call.** The row meaning you chose last week decides what every histogram in this notebook is
showing. A distribution of `energy` across 32,510 associations is not a distribution of
songs: a track on five playlists is in there five times. Neither is wrong. But you have to
know which one you are drawing, because the caption is different.


## 1. Two tables, two different pictures

`joined` has one row per association. `tracks` has one row per track. The same column has a
different distribution in each, and which one you want depends on the question.


In [ ]:
# The same column, counted two ways.
print("Associations:", len(joined))
print("Tracks:      ", len(tracks))
print()
print("Mean energy across associations:", round(joined["energy"].mean(), 4))
print("Mean energy across tracks:      ", round(tracks["energy"].mean(), 4))


Close, but not identical, and the gap is not noise. It is the tracks that appear on many
playlists being counted many times. For a question about *playlist placements*, the
association number is right. For a question about *songs*, it is not.


## 2. Distributions, and what a mean hides

`describe` is the fastest first look at a numeric column. It is also not enough on its own.


In [ ]:
# Step 1 - the numbers.
audio = ["energy", "danceability", "valence", "tempo", "duration_ms", "loudness"]
display(tracks[audio].describe().round(2))


In [ ]:
# Step 2 - the shapes. Four columns, four very different stories.
fig, axes = plt.subplots(2, 2, figsize=(11, 7))
for ax, column in zip(axes.flat, ["energy", "danceability", "valence", "tempo"]):
    tracks[column].plot(kind="hist", bins=40, ax=ax, color="#2f6f9f", edgecolor="white")
    ax.set(title=column, ylabel="tracks")
    ax.axvline(tracks[column].mean(), color="#d97941", linewidth=2)
plt.tight_layout()
plt.show()


The orange line is the mean of each column. Look at `valence` and at `tempo` before reading
on.

`valence` is spread almost flat across its whole range, so its mean sits in a region where
there is nothing special happening. `tempo` has a spike near 120 beats per minute and a
second bump around 100, because a lot of produced music is written to a small number of
conventional tempos. In both cases the mean is a real number and a poor summary.

`energy` is skewed towards the top of the range. `danceability` is the closest thing here to
a single hump.


> **Judgement call.** A histogram takes ten seconds and a language model will happily skip it, because nothing in
"compare the mean energy of two genres" asks for one. The mean of a bimodal column is a
number no row is near. Knowing to look is the skill; drawing it is not.


### The same information, five numbers at a time

A box plot is the compressed version: median, the middle half, and the tails. It is worth
less than a histogram for shape and more for comparing groups side by side, which is where
we are going in part 2.


In [ ]:
joined.boxplot(column="energy", by="genre", figsize=(9, 5), grid=False)
plt.suptitle("")
plt.title("Energy by genre, across associations")
plt.ylabel("energy")
plt.xlabel("playlist genre")
plt.show()


## 3. Rows that cannot be right

`describe` printed a minimum duration of 4,000 milliseconds. That is four seconds. Before
deciding what to do about it, look at the row.


In [ ]:
# Step 1 - the shortest tracks in the file.
display(tracks.nsmallest(4, "duration_ms")[["title", "artist", "duration_ms", "tempo", "energy"]])


In [ ]:
# Step 2 - a separate check, on a different column: a tempo of zero is not a slow song.
display(tracks.loc[tracks["tempo"] == 0, ["title", "artist", "tempo", "duration_ms", "energy"]])


It is the same row both times. A four-second track whose tempo is zero is not a short song
with no beat, it is a record that failed to be a song. One row, caught twice, by two checks
that had nothing to do with each other.

That is the useful part. You did not find it by suspecting it. You found it by looking at
the minimum of two different columns.


In [ ]:
# How much of the file is in question?
print("Tracks under one minute:", int((tracks["duration_ms"] < 60_000).sum()))
print("Tracks with tempo exactly 0:", int((tracks["tempo"] == 0).sum()))
print("Tracks louder than 0 dB:", int((tracks["loudness"] > 0).sum()))
print("Longest track, minutes:", round(tracks["duration_ms"].max() / 60_000, 1))


Twenty-five tracks under a minute, one with no tempo, six above 0 dB, and a longest track of
under nine minutes. So there is no crowd of broken rows here. There is a handful.

The honest move for this dataset is to leave them in, say that you looked, and note that 25
rows out of 28,356 cannot move a mean. If you were computing a median track length for a
contract, you would exclude them and say so.


> **Judgement call.** Nobody can tell you the right threshold for "too short to be a song". Four seconds is
obviously broken and three minutes is obviously fine, and somewhere between them is a line
that depends on what you are about to claim. That line is a decision you own and have to
defend, and it is the reason this cannot be a rule in a library.


## 4. Categories and base rates

Counting rows per category is the least glamorous cell in any analysis and it prevents more
mistakes than anything else in this notebook.


In [ ]:
genre_counts = joined["genre"].value_counts()
display(genre_counts.to_frame("associations").assign(
    share=(genre_counts / len(joined)).round(3)))


Reasonably balanced, between roughly 4,800 and 5,900 associations per genre. That is lucky,
and it is not the normal case. When one group has forty rows and another has forty thousand,
a difference in means between them says much more about sample size than about the world.

Keep this table nearby. Every summary from here on gets its counts printed next to it.


In [ ]:
# Subgenre is where the counts get thin. Same idea, one level down.
display(joined.groupby(["genre"])["track_id"].nunique().rename("distinct tracks").to_frame())


## 5. Relationships between columns

Correlation is one number describing how two columns move together. It is easy to compute
and easy to over-read, so we do the number and the picture together.


In [ ]:
# Step 1 - the whole matrix at once, on tracks rather than associations.
correlations = tracks[["energy", "danceability", "loudness", "valence", "tempo",
                       "acousticness", "popularity"]].corr()
display(correlations.round(2))


In [ ]:
# Step 2 - just the column we care about, sorted.
display(correlations["energy"].drop("energy").sort_values(ascending=False).round(3).to_frame())


Energy and loudness sit at about 0.68, which is high and unsurprising: both are partly
measuring how much is going on in the recording. Energy and acousticness are at about -0.55,
which is close to a definition rather than a discovery. Energy and danceability are at about
-0.08, which is nothing at all, and worth saying out loud because "energetic" and "danceable"
sound like they should go together in English.


In [ ]:
# Step 3 - look at the strong one. 28,000 points is too many to plot honestly.
sample = tracks.sample(2_000, random_state=2026)
ax = sample.plot(kind="scatter", x="loudness", y="energy", alpha=0.25, s=12,
                 figsize=(8, 5), color="#2f6f9f")
ax.set(title="Energy against loudness, 2,000 sampled tracks", xlabel="loudness (dB)", ylabel="energy")
plt.show()


Two things to take from the picture that the number 0.68 does not tell you. The relationship
is there but the cloud is wide, so loudness is not a stand-in for energy on any individual
track. And there is a tail of very quiet tracks stretching left, which is a different kind of
music rather than a measurement error.

Plotting all 28,356 points would have produced a solid block of ink. Sampling 2,000 with a
fixed `random_state` is honest and reproducible, and says so in the title.


> **Judgement call.** A correlation matrix will hand you fifteen numbers and no opinion about which of them are
worth a sentence. Energy against loudness at 0.68 is nearly a tautology. Energy against
danceability at -0.08 is the interesting one, because it contradicts what the words suggest.
Deciding which numbers are findings is the entire job, and it is not in the matrix.


## Practice

Two tasks. Try both before the solutions at the bottom.

**Practice 1.** Draw the distribution of `duration_ms` in minutes rather than milliseconds,
for tracks only. Say what the mean hides. Then decide, and write down, which tracks you would
exclude for a claim about typical song length.

**Practice 2.** Pick any two audio columns other than energy and loudness. Compute their
correlation, plot a sample of them, and write one sentence about whether the number and the
picture agree.


In [ ]:
# Practice 1. Your turn.
# 1. Build a duration-in-minutes column on tracks.
# 2. Plot it.
# 3. Print the mean and the median.
# 4. State your exclusion rule and how many rows it removes.


In [ ]:
# Practice 2. Your turn.
# 1. Pick two columns.
# 2. Correlation.
# 3. Scatter on a fixed-seed sample.
# 4. One sentence: do the number and the picture agree?


## Solutions


In [ ]:
# Solution 1.
tracks_minutes = tracks.assign(duration_min=tracks["duration_ms"] / 60_000)
ax = tracks_minutes["duration_min"].plot(kind="hist", bins=60, figsize=(9, 4),
                                         color="#2f6f9f", edgecolor="white")
ax.set(title="Track length in minutes", xlabel="minutes", ylabel="tracks")
plt.show()

print("Mean minutes:  ", round(tracks_minutes["duration_min"].mean(), 2))
print("Median minutes:", round(tracks_minutes["duration_min"].median(), 2))
short = tracks_minutes["duration_min"] < 1
print(f"Under one minute: {int(short.sum())} tracks, {short.mean():.2%} of the file")


Model answer: the mean is slightly above the median because a thin right tail of long tracks
pulls it up, and neither number tells you that the bulk of the file sits in a narrow band
between about three and four minutes. For a claim about typical song length, excluding
tracks under one minute is defensible and removes 25 rows, which cannot change the answer.
Saying that you checked is the part that matters.


In [ ]:
# Solution 2, one possible pair.
pair = ["valence", "danceability"]
print("Correlation:", round(tracks[pair[0]].corr(tracks[pair[1]]), 3))

sample = tracks.sample(2_000, random_state=2026)
ax = sample.plot(kind="scatter", x=pair[0], y=pair[1], alpha=0.25, s=12,
                 figsize=(7, 5), color="#187c80")
ax.set(title=f"{pair[1]} against {pair[0]}, 2,000 sampled tracks")
plt.show()


Model answer: valence and danceability correlate at about 0.33, which is a real but loose
relationship. The picture agrees and adds something the number does not: the cloud is wide
enough that you could not predict one from the other for any single track, which is the
difference between "these move together on average" and "these are the same thing".

## Where this goes next

Part 2 takes the box plot from section 2 and asks the obvious follow-up. Genres clearly
differ in energy, but how much of that is a real difference and how much is 32,000 rows
making everything look certain? That is where p-values arrive, and where they turn out to be
much less useful than they look.
